# StageBridge Training Tutorial

This notebook demonstrates how to train a StageBridge model on your own data.

**Training consists of two stages:**
1. **SSL Pretraining**: Learn niche-aware representations via masked receiver reconstruction
2. **Transition Model**: Learn stage transition dynamics via OT-CFM flow matching

**Requirements:**
```bash
pip install stagebridge torch scanpy
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

import stagebridge as sb
from stagebridge.models import StageBridge, StageBridgeConfig
from stagebridge.training import StageBridgeTrainer, TrainerConfig

# Check for GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 1. Prepare Data

First, load your data and prepare neighborhoods. See `01_quickstart.ipynb` for details.

In [ ]:
# Option 1: Load from parquet (recommended for large datasets)
# neighborhoods = pd.read_parquet("data/neighborhoods.parquet")

# Option 2: Load from AnnData and prepare neighborhoods
# import scanpy as sc
# adata = sc.read_h5ad("data/spatial.h5ad")
# neighborhoods = sb.prepare_neighborhoods(adata)

# Option 3: Create synthetic data for demonstration
from stagebridge.data.synthetic import generate_synthetic_dataset

neighborhoods, ground_truth = generate_synthetic_dataset(
    n_cells=5000,
    n_stages=3,
    stage_names=["Normal", "Preinvasive", "Invasive"],
    include_ground_truth=True,
)

print(f"Dataset size: {len(neighborhoods)}")
print(f"Stages: {neighborhoods['stage'].value_counts().to_dict()}")

## 2. Create DataLoaders

In [ ]:
from stagebridge.loaders import create_dataloaders

# Create train/val/test loaders with donor-held-out splits
train_loader, val_loader, test_loader = create_dataloaders(
    neighborhoods,
    batch_size=64,
    train_frac=0.7,
    val_frac=0.15,
    test_frac=0.15,
    num_workers=4,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader) if test_loader else 0}")

## 3. Configure Model

In [ ]:
# Model configuration
model_config = StageBridgeConfig(
    # Architecture
    input_dim=40,           # scVI latent dimension
    hidden_dim=256,         # Hidden dimension
    num_heads=8,            # Attention heads
    num_encoder_layers=3,   # Transformer layers
    
    # Key components (our novelty)
    use_cross_attn_drift=True,     # Cross-attention in drift head
    use_context_refiner=True,       # SetTransformer refiner
    use_learned_ring_pooling=True,  # Learned pooling per ring
    
    # Reference fusion
    use_gw_fusion=True,     # Gromov-Wasserstein fusion
    
    # Stage transitions
    num_stages=3,
)

# Create model
model = StageBridge(model_config)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4. Configure Training

In [ ]:
# Training configuration
trainer_config = TrainerConfig(
    # Output
    output_dir="runs/stagebridge",
    run_name="tutorial_run",
    
    # Two-stage training
    ssl_epochs=50,           # Stage 1: SSL pretraining
    transition_epochs=100,   # Stage 2: Flow matching
    freeze_encoder=False,    # Fine-tune encoder in stage 2
    
    # Learning rate
    learning_rate=1e-4,
    weight_decay=1e-5,
    warmup_epochs=5,
    
    # SSL loss weights
    ssl_reconstruction_weight=1.0,
    ssl_entropy_weight=0.01,
    
    # Transition loss weights  
    flow_matching_weight=1.0,
    pathway_weight=0.1,
    proliferation_weight=0.1,
    
    # OT-CFM settings
    use_ot=True,
    ot_epsilon=0.05,
    sinkhorn_iters=80,
    
    # Checkpointing
    checkpoint_every=10,
    keep_top_k=3,
    
    # Early stopping
    early_stopping_enabled=True,
    early_stopping_patience=15,
    
    # Hardware
    mixed_precision=True,
    gradient_clip=1.0,
)

print(f"Total epochs: {trainer_config.ssl_epochs + trainer_config.transition_epochs}")

## 5. Train Model

In [ ]:
# Create trainer
trainer = StageBridgeTrainer(
    model=model,
    config=trainer_config,
    device=device,
)

# Train (this will take a while!)
# For quick testing, reduce ssl_epochs and transition_epochs
summary = trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
)

print("\nTraining complete!")
print(f"Final SSL loss: {summary['ssl'].get('final_loss', 'N/A'):.4f}")
print(f"Final transition loss: {summary['transition'].get('final_loss', 'N/A'):.4f}")

## 6. Monitor Training

Training logs are saved to the output directory. You can visualize with TensorBoard:

```bash
tensorboard --logdir runs/stagebridge/tutorial_run/logs
```

Or load metrics directly:

In [ ]:
# Load training metrics
import json
from pathlib import Path

log_dir = Path(trainer_config.output_dir) / trainer_config.run_name / "logs"

if (log_dir / "metrics.json").exists():
    with open(log_dir / "metrics.json") as f:
        metrics = json.load(f)
    
    # Plot training curves
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    if "ssl_loss" in metrics:
        axes[0].plot(metrics["ssl_loss"], label="SSL Loss")
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel("Loss")
        axes[0].set_title("SSL Pretraining")
        axes[0].legend()
    
    if "transition_loss" in metrics:
        axes[1].plot(metrics["transition_loss"], label="Flow Loss")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Loss")
        axes[1].set_title("Transition Model")
        axes[1].legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("No metrics file found (training may not have completed)")

## 7. Load Best Checkpoint

In [ ]:
# Load the best checkpoint
checkpoint_dir = Path(trainer_config.output_dir) / trainer_config.run_name / "checkpoints"
best_checkpoint = checkpoint_dir / "best.pt"

if best_checkpoint.exists():
    model = sb.StageBridge.from_pretrained(best_checkpoint, device=device)
    print(f"Loaded best model from {best_checkpoint}")
else:
    print("Best checkpoint not found")
    # Use the model we just trained
    model = sb.StageBridge(trainer.model, model_config, device=device)

## 8. Evaluate on Test Set

In [ ]:
from stagebridge.evaluation import evaluate_reconstruction, evaluate_transitions

# Evaluate SSL reconstruction
if test_loader is not None:
    recon_metrics = evaluate_reconstruction(model._model, test_loader, device=device)
    print("Reconstruction metrics:")
    for k, v in recon_metrics.items():
        print(f"  {k}: {v:.4f}")

In [ ]:
# Evaluate transition predictions
# Compare predicted transitions with ground truth (if available)
if ground_truth is not None:
    print("\nGround truth evaluation available")
    print(f"  True velocity MSE: {ground_truth.get('velocity_mse', 'N/A')}")

## 9. Run Inference

In [ ]:
# Get embeddings for all cells
niche_output = model.embed_niches(neighborhoods, batch_size=256)

# Predict transitions
predictions = model.predict(
    neighborhoods=neighborhoods,
    source_stage="Normal",
    target_stage="Invasive",
)

print(f"Generated embeddings for {len(niche_output.embeddings)} cells")
print(f"Generated predictions for {len(predictions.predicted_embeddings)} cells")

In [ ]:
# Visualize results
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
emb_pca = pca.fit_transform(niche_output.embeddings)

stage_colors = {"Normal": "#1B4F72", "Preinvasive": "#2E86AB", "Invasive": "#922B21"}

fig, ax = plt.subplots(figsize=(8, 6))
for stage in ["Normal", "Preinvasive", "Invasive"]:
    mask = neighborhoods["stage"] == stage
    ax.scatter(emb_pca[mask, 0], emb_pca[mask, 1], 
               c=stage_colors[stage], label=stage, alpha=0.5, s=10)

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Learned Niche Embeddings")
ax.legend()
plt.show()

## Next Steps

- **Hyperparameter Tuning**: Use `stagebridge.pipelines.run_hpo` for Optuna-based HPO
- **Biological Analysis**: See `03_biological_interpretation.ipynb`
- **Multi-GPU Training**: Use PyTorch DDP with the trainer

### Recommended Hyperparameter Ranges

| Parameter | Range | Notes |
|-----------|-------|-------|
| hidden_dim | 128-512 | Larger for complex data |
| num_heads | 4-8 | More heads = more attention patterns |
| num_encoder_layers | 2-4 | Deeper = more expressivity |
| learning_rate | 1e-5 - 1e-3 | Start with 1e-4 |
| ot_epsilon | 0.01 - 0.1 | Lower = sharper couplings |